In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "utils" / "config.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find utils/config.py in the current directory or its parents. "
        "Set PROJECT_ROOT to the project repository root."
    )
sys.path.insert(0, str(PROJECT_ROOT))


import pandas as pd

from model.Backbone import *
from utils.config import *
from data.dataset_loader import load_split_csvs, load_data
from utils.train import evaluate
from utils.complexity import *

# Explicitly lock the requested evaluation configuration.
assert DATASET == "kermany"
assert SPLIT_SEED == 42

CSV_DIR = PROJECT_ROOT / "csv" / "kermany"
CHECKPOINT_PATH = PROJECT_ROOT / "model" / "weights" / "baseline_kermany42.pth"
assert CHECKPOINT_PATH.is_file(), f"Checkpoint not found: {CHECKPOINT_PATH}"

c:\Users\dehoa\Downloads\Models\Model Config\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-11-20 19:17:53 | Run: fe-res50-kermany-seed42
2026-11-20 19:17:53 | Config: {"run_name": "fe-res50-kermany-seed42", "dataset": "kermany", "split_seed": 42, "split_csv_dir": "./csv/kermany", "seed": 42, "use_mask": false, "input_masking": false, "feature_masking": false, "use_inverted_mask": false, "layers": 50, "use_eca": true, "use_dsa": false, "augment": false, "image_size": [224, 224], "batch_size": 128, "epochs": 30, "warmup_epochs": 10, "lr": 0.001, "weight_decay": 0.0001, "eval_checkpoint": "last", "focal_gamma": 2.0, "focal_alpha": null, "optimizer": "AdamW", "scheduler": "LinearLR(1e-6->1) warm-up + CosineAnnealing(eta_min=1e-6)"}
2026-11-20 19:17:53 | torch 2.15.0+cpu | device cpu | GPUs 0
2026-11-20 19:17:53 | Dataset 'kermany' downloaded to C:\Users\dehoa\.cache\kagglehub\datasets\paultimothymooney\chest-xray-pneumonia\versions\2


## Model

In [2]:
import copy
import math
from typing import Optional
import torch
import torch.nn.functional as F
from torch import Tensor, nn

class PositionEmbeddingSine(nn.Module):
    """
    This is a more standard version of the position embedding, very similar to the one
    used by the Attention is all you need paper, generalized to work on images.
    """

    def __init__(self, num_pos_feats=64, temperature=10000, normalize=False, scale=None):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, x, mask=None):
        if mask is None:
            mask = torch.zeros((x.size(0), x.size(2), x.size(3)), device=x.device, dtype=torch.bool)
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        if self.normalize:
            eps = 1e-6
            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)

        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack(
            (pos_x[:, :, :, 0::2].sin(), pos_x[:, :, :, 1::2].cos()), dim=4
        ).flatten(3)
        pos_y = torch.stack(
            (pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4
        ).flatten(3)
        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)
        return pos

class Transformer(nn.Module):
    def __init__(
        self,
        d_model=512,
        nhead=8,
        num_encoder_layers=6,
        num_decoder_layers=6,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
        return_intermediate_dec=False,
    ):
        super().__init__()

        encoder_layer = TransformerEncoderLayer(
            d_model, nhead, dim_feedforward, dropout, activation, normalize_before
        )
        encoder_norm = nn.LayerNorm(d_model) if normalize_before else None
        self.encoder = TransformerEncoder(encoder_layer, num_encoder_layers, encoder_norm)

        decoder_layer = TransformerDecoderLayer(
            d_model, nhead, dim_feedforward, dropout, activation, normalize_before
        )
        decoder_norm = nn.LayerNorm(d_model)
        self.decoder = TransformerDecoder(
            decoder_layer,
            num_decoder_layers,
            decoder_norm,
            return_intermediate=return_intermediate_dec,
        )

        self._reset_parameters()

        self.d_model = d_model
        self.nhead = nhead

    def _reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, mask, query_embed, query_pos, value_pos):
        # flatten NxCxHxW to HWxNxC
        bs, c, l = query_embed.shape
        query_embed = query_embed.permute(2, 0, 1)
        query_pos = query_pos.unsqueeze(0).expand(bs,-1,-1).permute(2, 0, 1)        
        value=src.flatten(2).permute(2, 0, 1)
        value_pos=value_pos.flatten(2).permute(2, 0, 1)
        if mask is not None:
            mask = mask.flatten(1)

        #tgt = torch.zeros_like(query_embed)
        memory = self.encoder(value, src_key_padding_mask=mask, pos=value_pos)
        hs = self.decoder(
            tgt=query_embed, memory=memory, memory_key_padding_mask=mask, pos=value_pos, query_pos=query_pos
        )
        return hs.transpose(1, 2).transpose(0, 2)


class TransformerEncoder(nn.Module):
    def __init__(self, encoder_layer, num_layers, norm=None):
        super().__init__()
        self.layers = _get_clones(encoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm

    def forward(
        self,
        src,
        mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        output = src

        for layer in self.layers:
            output = layer(
                output, src_mask=mask, src_key_padding_mask=src_key_padding_mask, pos=pos
            )

        if self.norm is not None:
            output = self.norm(output)

        return output


class TransformerDecoder(nn.Module):
    def __init__(self, decoder_layer, num_layers, norm=None, return_intermediate=False):
        super().__init__()
        self.layers = _get_clones(decoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm
        self.return_intermediate = return_intermediate

    def forward(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
    ):
        output = tgt

        intermediate = []

        for layer in self.layers:
            output = layer(
                tgt=output,
                memory=memory,
                tgt_mask=tgt_mask,
                memory_mask=memory_mask,
                tgt_key_padding_mask=tgt_key_padding_mask,
                memory_key_padding_mask=memory_key_padding_mask,
                pos=pos,
                query_pos=query_pos,
            )
            if self.return_intermediate:
                intermediate.append(self.norm(output))

        if self.norm is not None:
            output = self.norm(output)
            if self.return_intermediate:
                intermediate.pop()
                intermediate.append(output)

        if self.return_intermediate:
            return torch.stack(intermediate)

        return output


class TransformerEncoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
    ):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward_post(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        q = k = self.with_pos_embed(src, pos)
        src2 = self.self_attn(
            q, k, value=src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask
        )[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

    def forward_pre(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        src2 = self.norm1(src)
        q = k = self.with_pos_embed(src2, pos)
        src2 = self.self_attn(
            q, k, value=src2, attn_mask=src_mask, key_padding_mask=src_key_padding_mask
        )[0]
        src = src + self.dropout1(src2)
        src2 = self.norm2(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src2))))
        src = src + self.dropout2(src2)
        return src

    def forward(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        if self.normalize_before:
            return self.forward_pre(src, src_mask, src_key_padding_mask, pos)
        return self.forward_post(src, src_mask, src_key_padding_mask, pos)


class TransformerDecoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
    ):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

        self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward_post(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
    ):
        q = k = self.with_pos_embed(tgt, query_pos)
        tgt2 = self.self_attn(
            q, k, value=tgt, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask
        )[0]
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)
        tgt2 = self.multihead_attn(
            query=self.with_pos_embed(tgt, query_pos),
            key=self.with_pos_embed(memory, pos),
            value=memory,
            attn_mask=memory_mask,
            key_padding_mask=memory_key_padding_mask,
        )[0]
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        tgt = tgt + self.dropout3(tgt2)
        tgt = self.norm3(tgt)
        return tgt

    def forward_pre(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
    ):
        tgt2 = self.norm1(tgt)
        q = k = self.with_pos_embed(tgt2, query_pos)
        tgt2 = self.self_attn(
            q, k, value=tgt2, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask
        )[0]
        tgt = tgt + self.dropout1(tgt2)
        tgt2 = self.norm2(tgt)
        tgt2 = self.multihead_attn(
            query=self.with_pos_embed(tgt2, query_pos),
            key=self.with_pos_embed(memory, pos),
            value=memory,
            attn_mask=memory_mask,
            key_padding_mask=memory_key_padding_mask,
        )[0]
        tgt = tgt + self.dropout2(tgt2)
        tgt2 = self.norm3(tgt)
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt2))))
        tgt = tgt + self.dropout3(tgt2)
        return tgt

    def forward(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
    ):
        if self.normalize_before:
            return self.forward_pre(
                tgt,
                memory,
                tgt_mask,
                memory_mask,
                tgt_key_padding_mask,
                memory_key_padding_mask,
                pos,
                query_pos,
            )
        return self.forward_post(
            tgt,
            memory,
            tgt_mask,
            memory_mask,
            tgt_key_padding_mask,
            memory_key_padding_mask,
            pos,
            query_pos,
        )


def _get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])


def _get_activation_fn(activation):
    """Return an activation function given a string"""
    if activation == "relu":
        return F.relu
    if activation == "gelu":
        return F.gelu
    if activation == "glu":
        return F.glu
    raise RuntimeError(f"activation should be relu/gelu, not {activation}.")

In [3]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction_ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // reduction_ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)
        attention = self.sigmoid(self.conv(concat))
        return attention

class CRAM(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16, kernel_size=7):
        super(CRAM, self).__init__()
        self.channel_att = ChannelAttention(in_channels, reduction_ratio)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x

class FeatureEnhancementBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(FeatureEnhancementBlock, self).__init__()
        self.cram = CRAM(in_channels)
        self.residual = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels)
        ) 
        
    def forward(self, x):
        residual = self.residual(x)
        x = self.cram(x)
        return x + residual

class MSR(nn.Module):
    def __init__(self, layers, num_classes=2, reduce_dim=256): 
        super(MSR, self).__init__()
        self.backbone = Backbone(
            'resnet{}'.format(layers), 
            train_backbone=False, 
            return_interm_layers=True, 
            dilation=[False, True, True]
        )
        
        # Feature enhancement modules - FIXED CHANNEL DIMENSIONS
        self.enhancement_cat = FeatureEnhancementBlock(reduce_dim, reduce_dim)  # Changed from (512+1024, reduce_dim)
        self.enhancement_3 = FeatureEnhancementBlock(reduce_dim, reduce_dim)    # Changed from (2048, reduce_dim)
        
        self.embed_cat = nn.Embedding(reduce_dim, 1) 
        self.embed_3 = nn.Embedding(reduce_dim, 1) 
        self.pe_layer_cat = PositionEmbeddingSine(reduce_dim//2, normalize=True)
        self.pe_layer_3 = PositionEmbeddingSine(reduce_dim//2, normalize=True)

        self.transformer_cat = Transformer(
            d_model=reduce_dim,
            dropout=0.1,
            nhead=4,
            dim_feedforward=reduce_dim//4,
            num_encoder_layers=0,
            num_decoder_layers=1,
            normalize_before=False,
            return_intermediate_dec=False,
        )

        self.transformer_3 = Transformer(
            d_model=reduce_dim,
            dropout=0.1,
            nhead=4,
            dim_feedforward=reduce_dim//4,
            num_encoder_layers=0,
            num_decoder_layers=1,
            normalize_before=False,
            return_intermediate_dec=False,
        )
        
        # Convolution layers for feature reduction
        self.conv_red_1 = nn.Sequential(
            nn.Conv2d(512, reduce_dim//2, kernel_size=1, padding=0, bias=False),
            nn.BatchNorm2d(reduce_dim//2),
            nn.ReLU()
        )
        self.conv_red_2 = nn.Sequential(
            nn.Conv2d(1024, reduce_dim//2, kernel_size=1, padding=0, bias=False),
            nn.BatchNorm2d(reduce_dim//2),
            nn.ReLU()
        )
        self.conv_red_3 = nn.Sequential(
            nn.Conv2d(2048, reduce_dim, kernel_size=1, padding=0, bias=False),
            nn.BatchNorm2d(reduce_dim),
            nn.ReLU()
        )

        # Cross-scale attention fusion
        self.cross_scale_attention = nn.MultiheadAttention(reduce_dim, 4, batch_first=True)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Enhanced classifier
        self.classifier = nn.Sequential(
            nn.Linear(reduce_dim * 2, reduce_dim),
            nn.BatchNorm1d(reduce_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(reduce_dim, reduce_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(reduce_dim // 2, num_classes)
        )
        
        self.num_classes = num_classes

    def forward(self, x):
        # Backbone feature extraction
        back_x = self.backbone(x)
        
        # Process individual feature maps
        red_back_1 = self.conv_red_1(back_x['1'])
        red_back_2 = self.conv_red_2(back_x['2'])
        red_back_3 = self.conv_red_3(back_x['3'])
        
        # Concatenate features (now both have reduce_dim channels)
        red_back_cat = torch.cat((red_back_1, red_back_2), dim=1)
        
        # Apply feature enhancement blocks
        enhanced_cat = self.enhancement_cat(red_back_cat)
        enhanced_3 = self.enhancement_3(red_back_3)
        
        # Prepare for transformer
        avg_back_cat = self.avgpool(enhanced_cat)
        avg_back_3 = self.avgpool(enhanced_3)

        masking = None
        query_pos = self.embed_cat.weight

        # Transformer processing
        key_embed = enhanced_cat
        query_embed = avg_back_cat.squeeze(-1)
        key_pos = self.pe_layer_cat(enhanced_cat)
        fg_embed_cat = self.transformer_cat(key_embed, masking, query_embed, query_pos, key_pos)
        
        query_pos = self.embed_3.weight
        key_embed = enhanced_3
        query_embed = avg_back_3.squeeze(-1)
        key_pos = self.pe_layer_3(enhanced_3)
        fg_embed_3 = self.transformer_3(key_embed, masking, query_embed, query_pos, key_pos)
        
        # Final concatenation and classification
        out = torch.cat((fg_embed_cat, fg_embed_3), dim=1)     
        out_1 = torch.flatten(out, 1)
        out_1 = self.classifier(out_1)
        
        return out_1

    def get_backbone_params(self):
        return self.backbone.parameters()

    def get_fc_params(self):
        return self.classifier.parameters()

In [4]:
NUM_CLASSES = 2
model = MSR(
    layers=LAYERS,
    num_classes=NUM_CLASSES,
).to(device)

c:\Users\dehoa\Downloads\Models\Model Config\myenv\Lib\site-packages\torchvision\models\_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\dehoa\Downloads\Models\Model Config\myenv\Lib\site-packages\torchvision\models\_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [10]:
def load_original_checkpoint_into_revised_model(model, checkpoint_path):
    state = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    state = {
        k.removeprefix("module."): v
        for k, v in state.items()
    }
    current_state = model.state_dict()
    missing = sorted(set(current_state) - set(state))
    unexpected = sorted(set(state) - set(current_state))

    mismatched = [
        (key, tuple(state[key].shape), tuple(current_state[key].shape))
        for key in set(state) & set(current_state)
        if state[key].shape != current_state[key].shape
    ]
    print(f"Missing keys: {len(missing)}")
    for key in missing:
        print(f"  {key}")
    print(f"\nUnexpected keys: {len(unexpected)}")
    for key in unexpected:
        print(f"  {key}")
    print(f"\nShape mismatches: {len(mismatched)}")
    for item in mismatched:
        print(f"  {item}")

    # Stop if required model weights are missing or incompatible.
    if missing or mismatched:
        raise RuntimeError("Checkpoint validation failed. Weights were not loaded.")

    model.load_state_dict(state, strict=False)
    print("\nCheckpoint loaded successfully.")
    return model

model = load_original_checkpoint_into_revised_model(model, CHECKPOINT_PATH)
model = model.to(device)
model.eval()
model.eval()


Missing keys: 0

Unexpected keys: 0

Shape mismatches: 0

Checkpoint loaded successfully.


MSR(
  (backbone): Backbone(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d()
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d()
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d()
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d()
          (relu): ReLU(inplace=True)
          (downsample): Sequential(
            (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): FrozenBatchNorm2d()
          )
        )
        (1): Bottleneck(
          (conv1): Conv

In [11]:
# Load the exact saved train/val/test CSVs; use only the test loader for evaluation.
splits = load_split_csvs(
    csv_dir=CSV_DIR,
    dataset="kermany",
    seed=SPLIT_SEED,
    root=str(EXPECTED_DIR),
)
train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]

# load_data expects all three frames but constructs the test loader with val transforms
_, _, test_loader = load_data(train_df, val_df, test_df)

print("Test images:", len(test_df))
print("Class counts:")
print(test_df["class_name"].value_counts().sort_index())


2026-11-20 19:25:10 | Loaders: train 4684 | val 586 | test 586 | batch 128 | masks OFF
Test images: 586
Class counts:
class_name
NORMAL       158
PNEUMONIA    428
Name: count, dtype: int64


In [12]:
class FocalLossCE(nn.Module):
    """Focal loss no class weighting (== alpha 1)."""
    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        super().__init__()
        self.gamma, self.alpha, self.reduction = gamma, alpha, reduction

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.alpha is not None:
            loss = self.alpha[targets] * loss
        return loss.mean() if self.reduction == "mean" else loss.sum()

In [ ]:
criterion = FocalLossCE()
test_loss, test_accuracy, y_true, y_pred, y_prob = evaluate(
    model,
    test_loader,
    criterion,
    device,
    invert_mask=False,
    feature_masking=False,
    input_masking=False,
)

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

print(f"Test loss: {test_loss:.6f}")
print(f"Test accuracy: {test_accuracy:.4f}%")
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=sorted(test_df["class_name"].unique()), digits=4))

results = pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred,
    "prob_class_0": y_prob[:, 0],
    "prob_class_1": y_prob[:, 1],
})
compute_complexity(model)

c:\Users\dehoa\Downloads\Models\Model Config\myenv\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Test loss: 0.024078
Test accuracy: 96.7577%
Confusion matrix:
[[147  11]
 [  8 420]]
              precision    recall  f1-score   support

      NORMAL     0.9484    0.9304    0.9393       158
   PNEUMONIA     0.9745    0.9813    0.9779       428

    accuracy                         0.9676       586
   macro avg     0.9614    0.9558    0.9586       586
weighted avg     0.9674    0.9676    0.9675       586

2026-11-20 19:27:09 | Total parameters: 28,107,398 | trainable: 4,652,486
2026-11-20 19:27:09 | Inference time (1 image, CPU, single pass): 316.1 ms
Total:21.7 GMac
Module:  Global
aten.convolution: 21.49 GMac
aten.addmm: 1.02 MMac
aten.bmm: 803.84 KMac
aten.mm: 205.52 MMac

Module:  backbone
aten.convolution: 19.08 GMac

Module:  conv_red_1
aten.convolution: 51.38 MMac

Module:  conv_red_2
aten.convolution: 102.76 MMac

Module:  conv_red_3
aten.convolution: 411.04 MMac

Module:  enhancement_cat
aten.convolution: 925.34 MMac

Module:  enhancement_3
aten.convolution: 925.34 MMac

Mo